In [ ]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

from dotenv import load_dotenv
load_dotenv()  # loads PINECONE_API_KEY etc. from the .env file in this same folder

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
from langchain_classic.chains import HypotheticalDocumentEmbedder, LLMChain

from pinecone import Pinecone

from src.backend.logger import GLOBAL_LOGGER as log
from src.backend.core.config import settings
from src.backend.rag.embeddings import get_embeddings

****Data ingestion****

In [2]:
def load_document(directory_path):
    try:
        documents = []
        for filename in os.listdir(directory_path):
            file_path = os.path.join(directory_path, filename)
            if filename.endswith(".pdf"):
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".docx"):
                # First: extract normal docx text
                loader = Docx2txtLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".txt"):
                loader = TextLoader(file_path, encoding="utf-8")
                documents.extend(loader.load())

        return documents
    except Exception as e:
        log.error("Error loading documents from directory", error=str(e), directory=directory_path)
        raise e

docs = load_document("./TempData")
docs

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-14T08:45:46+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-12-14T08:45:46+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './TempData\\Acme_FY2024_UltraDense_Report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='ACME MANUFACTURING LTD – FY2024 CONSOLIDATED REPORT\nAcme Manufacturing Ltd is a multinational industrial manufacturing company operating across the United States, Germany, and the\nUnited Kingdom. The company manufactures heavy machinery, automotive components, and precision-engineered industrial\nequipment for aerospace and defense sectors. Acme’s customers include original equipment manufacturers, government agencies,\nand industrial distributors. The company prepares consolidated financial statements in accordance with International Financial\nReporting Standards 

****Embeddings****

In [3]:
embeddings = get_embeddings("OpenAI (text-embedding-3-small)")

In [ ]:
pinecone_api_key = os.getenv("PINECONE_API_KEY")
if not pinecone_api_key:
    raise RuntimeError("PINECONE_API_KEY not set -- check the .env file in this folder")
pc = Pinecone(api_key=pinecone_api_key)

****Create Index in Pinecone****

In [5]:
from pinecone import ServerlessSpec

index_name = "multi-document-assist"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

****Vector Store****

In [6]:
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

****Adding Documents to Pinecone Vector Store****

In [7]:
vector_store.add_documents(documents=docs)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


['428ffa8e-ecfd-4e5c-af9e-bde8db50ff50',
 'bae80fd6-0e1e-41ea-bb48-2ca57e787e96']

****Retreival Phase****

In [8]:
results = vector_store.similarity_search(
    "what is the revenue increase in FY24?",
    k=1
)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [9]:
results

[Document(id='bae80fd6-0e1e-41ea-bb48-2ca57e787e96', metadata={'author': '(anonymous)', 'creationdate': '2025-12-14T08:45:46+00:00', 'creator': '(unspecified)', 'keywords': '', 'moddate': '2025-12-14T08:45:46+00:00', 'page': 1.0, 'page_label': '2', 'producer': 'ReportLab PDF Library - www.reportlab.com', 'source': './TempData\\Acme_FY2024_UltraDense_Report.pdf', 'subject': '(unspecified)', 'title': '(anonymous)', 'total_pages': 2.0, 'trapped': '/False'}, page_content='BALANCE SHEET, COMPLIANCE & RISK DISCLOSURES\nBALANCE SHEET SUMMARY (FY2024): Total assets amounted to USD 200.0 million, comprising current assets of USD 50.0\nmillion and non-current assets of USD 150.0 million. Total liabilities stood at USD 140.0 million, including short-term borrowings of\nUSD 32.0 million and long-term debt of USD 80.0 million. Total equity attributable to shareholders amounted to USD 60.0 million.\nBalance Sheet (USD)\nFY2024\nCash & Equivalents\n18,000,000\nAccounts Receivable\n22,000,000\nInvento